# 02 - Bronze ingestion

Loads the raw CSV from the volume and writes it to a Delta table in the bronze schema as-is.
No cleaning or schema changes happen here - that's silver's job.


In [0]:
import sys
import os

# Make repo root importable so we can use utils/
sys.path.append(os.path.abspath("../.."))

from utils.constants import RAW_CSV_PATH, BRONZE_RESULTS_TABLE
from utils.io_helpers import read_csv_from_volume, write_df_to_table

## Read the CSV and write it to bronze

Reading the raw CSV from the volume and writing it straight to a Delta table in the bronze schema.
No transformations - bronze should be an exact copy of the source.

In [0]:
df = read_csv_from_volume(spark, RAW_CSV_PATH)
write_df_to_table(df, BRONZE_RESULTS_TABLE)

## Verify

Quick sanity check that the table was created and has the expected number of rows.

In [0]:
bronze_df = spark.table(BRONZE_RESULTS_TABLE)
print(f"Rows in bronze: {bronze_df.count():,}")
bronze_df.show(3, vertical=True, truncate=False)

In [0]:
from pyspark.sql import functions as F
from utils.constants import BRONZE_RESULTS_TABLE

bronze_df = spark.table(BRONZE_RESULTS_TABLE)
bronze_df.groupBy("Event distance/length").count().orderBy(F.col("count").desc()).show(30, truncate=False)

In [0]:
bronze_df.filter(F.col("Athlete performance").contains(":")) \
    .select("Event distance/length", "Athlete performance") \
    .show(20, truncate=False)

In [0]:
%reload_ext autoreload
%autoreload 2

from utils.cleaning import split_event_distance, split_performance, add_validity_flag

In [0]:
from pyspark.sql import functions as F
from utils.constants import BRONZE_RESULTS_TABLE
bronze_df = spark.table(BRONZE_RESULTS_TABLE)

test = split_event_distance(bronze_df)
test = split_performance(test)

test.select(
    "Athlete performance", "perf_unit", "perf_distance", "perf_seconds"
).show(10, truncate=False)